# 01 · Justificación del workflow (REQ-02)

**Pregunta:** ¿la recepción de disputas de transacciones es el workflow con más evidencia y valor? (§8.1)
Criterio de salida: si disputas/transacciones son marginales frente a otra razón de contacto → Card-Service Support.

Salida: `analysis/results/workflow_justification.json` (agregados) + figuras en `analysis/figures/`.

1. Razones de contacto y volumen · 2. Patrones de demanda · 3. Disputas dentro de las quejas ·
4. Ligar quejas con transacciones (riesgo §14) · 5. Calidad de datos · 6. Restricciones operativas · 7. Conclusión

In [ ]:
from common import ROOT, connect, save_result, show

con = connect()

## 1. Razones de contacto y volumen

In [ ]:
reasons = show(con, "Interacciones por categoría (+ escalación, resolución)", """
    SELECT reason_category, count(*) AS n, round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS pct,
           round(avg(was_escalated::INT), 3) AS p_escalado, round(avg(was_resolved::INT), 3) AS p_resuelto,
           round(median(duration_seconds)) AS dur_p50_s
    FROM call_center_interactions GROUP BY 1 ORDER BY n DESC""")

## 2. Patrones de demanda (mensual, por país/canal)

In [ ]:
monthly = show(con, "Quejas por mes y categoría", """
    SELECT date_trunc('month', creation_date::TIMESTAMP) AS mes, category, count(*) AS n
    FROM complaints GROUP BY ALL ORDER BY mes, category""", 60)

## 3. Disputas dentro de las quejas
`Cargo no reconocido` + `Cobro indebido` = disputas de transacción en nuestro alcance.

In [ ]:
disputes = show(con, "Subcategorías de quejas", """
    SELECT subcategory, count(*) AS n, round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS pct,
           round(avg(sla_breached::INT), 3) AS p_sla_roto, round(avg(is_repeat_complainer::INT), 3) AS p_reincidente,
           round(median(resolution_days), 1) AS dias_p50
    FROM complaints GROUP BY 1 ORDER BY n DESC""")

## 4. ¿Se pueden ligar quejas con transacciones? (§14)

No. El producto afectado nunca es del cliente que reclama y el monto reclamado coincide con sus transacciones al
nivel del azar (detalle y cifras en `02_policy_calibration`). La disputa se ancla a la transacción que el cliente
identifica en la conversación y que la API verifica como suya (POL-1), no a un vínculo histórico del dataset.

In [ ]:
import json

calibration = json.loads((ROOT / "analysis" / "results" / "policy_calibration.json").read_text(encoding="utf-8"))
link = calibration["dispute_transaction_link"]
link

## 5. Calidad de datos
Nulos, duplicados y huérfanos en las columnas del workflow (complementa `data/_state/reports/latest.json` de OPS).

In [ ]:
show(con, "Nulos en columnas clave de complaints", """
    SELECT round(avg((claimed_amount IS NULL)::INT), 3) AS claimed_amount_nulo,
           round(avg((affected_product_id IS NULL)::INT), 3) AS producto_nulo,
           round(avg((subcategory IS NULL)::INT), 3) AS subcategoria_nula,
           count(*) - count(DISTINCT complaint_id) AS duplicados
    FROM complaints""")

## 6. Restricciones operativas
SLA, tiempos de resolución y carga por agente: qué parte se automatiza sin mover dinero (CON-05).

In [ ]:
# TODO DS: SLA incumplido y días de resolución por prioridad; carga de service_agents.

## 7. Conclusión
Decisión con cifras; se guardan los agregados que cita el README.

In [ ]:
save_result("workflow_justification", {
    "reason_categories": reasons.to_dicts(),
    "complaint_subcategories": disputes.to_dicts(),
    "decision": "TODO: disputas de transacciones / criterio de salida",
})